In [ ]:
# =============================================================================
# CELL 1 — INSTALL + RESTART  (run this FIRST on every new session)
# Upgrades transformers to the version required by Qwen2.5 and phi-2.
# On second run: detects correct version already loaded, skips restart.
# =============================================================================
import subprocess, sys, os

def _ok():
    try:
        from transformers.utils.generic import merge_with_config_defaults
        return True
    except ImportError:
        return False

if not _ok():
    print("Upgrading packages — kernel will restart in a moment ...")
    subprocess.run([
        sys.executable, "-m", "pip", "install", "-q",
        "transformers>=4.51.0", "peft>=0.9.0",
        "accelerate>=0.26.0",  "bitsandbytes>=0.46.1",
        "duckdb", "gradio", "datasets", "gdown",
        "matplotlib", "seaborn",
    ], check=True)
    print("Done. Restarting kernel now ...")
    os.kill(os.getpid(), 9)          # Colab auto-reconnects after this
else:
    import transformers
    print(f"transformers {transformers.__version__} already correct.")
    print("Proceed to Cell 2.")


Upgrading packages — kernel will restart in a moment ...


In [1]:
# =============================================================================
# CELL 2 — MOUNT DRIVE + IMPORTS
# =============================================================================
from google.colab import drive
drive.mount('/content/drive')

import os, re, ast, random, textwrap, shutil
from datetime import datetime

import numpy as np
import pandas as pd
import torch
import duckdb
import matplotlib.pyplot as plt
import gradio as gr

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer
from pandas.api.types import is_numeric_dtype, is_datetime64_any_dtype

from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig,
    DataCollatorForLanguageModeling, Trainer, TrainingArguments,
)
from peft import LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training

import transformers
print(f"transformers : {transformers.__version__}")
print(f"torch        : {torch.__version__}")
print(f"CUDA         : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU          : {torch.cuda.get_device_name(0)}")


Mounted at /content/drive
transformers : 5.0.0
torch        : 2.10.0+cu128
CUDA         : True
GPU          : Tesla T4


In [2]:
# =============================================================================
# CELL 3 — LOAD DATA
# =============================================================================
FILE_PATH = "/content/drive/MyDrive/Capstone/Credit Card Data.csv"

df = pd.read_csv(FILE_PATH)
df['trans_date_trans_time'] = pd.to_datetime(df['trans_date_trans_time'])
df['dob']                   = pd.to_datetime(df['dob'])
df['age']          = df['trans_date_trans_time'].dt.year - df['dob'].dt.year
df['tx_hour']      = df['trans_date_trans_time'].dt.hour
df['tx_dayofweek'] = df['trans_date_trans_time'].dt.dayofweek
df = df.sort_values(['cc_num', 'trans_date_trans_time']).reset_index(drop=True)

print(f"Rows: {len(df):,}  |  Fraud rate: {df['is_fraud'].mean():.2%}")
df.head(3)


Rows: 1,296,675  |  Fraud rate: 0.58%


,Unnamed: 0,trans_date_trans_time,cc_num,merchant,category,amt,first,last,gender,street,...,dob,trans_num,unix_time,merch_lat,merch_long,is_fraud,merch_zipcode,age,tx_hour,tx_dayofweek
0,1017,2019-01-01 12:47:15,60416207185,"fraud_Jones, Sawayn and Romaguera",misc_net,7.27,Mary,Diaz,F,9886 Anita Drive,...,1986-02-17,98e3dcf98101146a577f85a34e58feec,1325422035,43.974711,-109.741904,0,NaN,33,12,1
1,2724,2019-01-02 08:44:57,60416207185,fraud_Berge LLC,gas_transport,52.94,Mary,Diaz,F,9886 Anita Drive,...,1986-02-17,498120fc45d277f7c88e3dba79c33865,1325493897,42.018766,-109.044172,0,NaN,33,8,2
2,2726,2019-01-02 08:47:36,60416207185,fraud_Luettgen PLC,gas_transport,82.08,Mary,Diaz,F,9886 Anita Drive,...,1986-02-17,95f514bb993151347c7acdf8505c3d62,1325494056,42.961335,-109.157564,0,NaN,33,8,2


In [3]:
# =============================================================================
# CELL 4 — USER PROFILING
# =============================================================================
def build_user_profiles(df_legit):
    agg = df_legit.groupby('cc_num').agg(
        n_legit_tx   =('amt','count'), total_spent  =('amt','sum'),
        mean_spent   =('amt','mean'),  std_spent    =('amt','std'),
        min_spent    =('amt','min'),   max_spent    =('amt','max'),
        median_spent =('amt','median'),age_mean     =('age','mean'),
        city_pop_mean=('city_pop','mean'),
    )
    cat_cnt = df_legit.groupby(['cc_num','category'])['amt'].count().unstack(fill_value=0)
    cat_pct = cat_cnt.div(cat_cnt.sum(axis=1), axis=0)
    cat_pct.columns = [f"cat_pct_{c}" for c in cat_pct.columns]

    def _tb(h):
        if h < 6:  return 'night'
        if h < 12: return 'morning'
        if h < 18: return 'afternoon'
        return 'evening'
    dl = df_legit.copy()
    dl['time_bucket'] = dl['tx_hour'].apply(_tb)
    tb_cnt = dl.groupby(['cc_num','time_bucket'])['amt'].count().unstack(fill_value=0)
    tb_pct = tb_cnt.div(tb_cnt.sum(axis=1), axis=0)
    tb_pct.columns = [f"tb_pct_{c}" for c in tb_pct.columns]

    geo = df_legit.groupby('cc_num').agg(
        main_state=('state', lambda x: x.value_counts().idxmax()),
        main_city =('city',  lambda x: x.value_counts().idxmax()),
        mean_lat  =('lat',  'mean'),
        mean_long =('long', 'mean'),
    )
    return agg.join(cat_pct).join(tb_pct).join(geo).reset_index()

df_legit      = df[df['is_fraud'] == 0].copy()
user_profiles = build_user_profiles(df_legit)

def profile_to_text(row):
    cat_cols = [c for c in user_profiles.columns if c.startswith('cat_pct_')]
    top_cats = sorted(
        [(c.replace('cat_pct_',''), row[c]) for c in cat_cols],
        key=lambda x: x[1], reverse=True)[:3]
    cat_str = ", ".join([f"{n} ({p:.0%})" for n, p in top_cats if p > 0])
    txt = (
        f"User {int(row['cc_num'])} lives mainly in "
        f"{row.get('main_city','?')}, {row.get('main_state','?')}. "
        f"They have {int(row.get('n_legit_tx',0))} legitimate transactions. "
        f"The average transaction amount is ${row.get('mean_spent',0):.2f} "
        f"and max is ${row.get('max_spent',0):.2f}. "
    )
    if cat_str:
        txt += f"The main spending categories are: {cat_str}. "
    return txt

user_profiles['profile_text'] = user_profiles.apply(profile_to_text, axis=1)
print(f"User profiles: {len(user_profiles):,}")


User profiles: 908


In [4]:

# =============================================================================
# CELL 5 — FEATURE ENGINEERING & TRAIN/TEST SPLIT
# =============================================================================
df_model = df.merge(user_profiles, on='cc_num', how='left', suffixes=('','_prof'))

feature_cols_cat = ['category','state','city','job','gender']
feature_cols_num = (
    ['amt','age','city_pop','tx_hour','tx_dayofweek',
     'n_legit_tx','total_spent','mean_spent','std_spent',
     'min_spent','max_spent','median_spent','mean_lat','mean_long']
    + [c for c in df_model.columns
       if c.startswith('cat_pct_') or c.startswith('tb_pct_')]
)

df_model_enc = df_model.copy()
label_encoders = {}
for col in feature_cols_cat:
    le = LabelEncoder()
    le.fit(np.append(df_model_enc[col].astype(str).unique(), 'UNKNOWN_CATEGORY'))
    df_model_enc[col] = le.transform(df_model_enc[col].astype(str))
    label_encoders[col] = le

X = df_model_enc[feature_cols_num + feature_cols_cat]
y = df_model_enc['is_fraud'].astype(int)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Train: {len(X_train):,}  Test: {len(X_test):,}")


Train: 1,037,340  Test: 259,335


In [5]:
# =============================================================================
# CELL 6 — BASELINE ML MODELS
# =============================================================================
imputer = SimpleImputer(strategy='median')
X_train_imputed = pd.DataFrame(
    imputer.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
X_test_imputed = pd.DataFrame(
    imputer.transform(X_test), columns=X_test.columns, index=X_test.index)

models_baseline = {
    "RandomForest"      : RandomForestClassifier(
        n_estimators=200, n_jobs=-1, random_state=42, class_weight='balanced'),
    "GradientBoosting"  : GradientBoostingClassifier(random_state=42),
    "LogisticRegression": LogisticRegression(max_iter=1000, class_weight='balanced'),
}
baseline_results, fpr_baseline = {}, {}

for name, clf in models_baseline.items():
    print(f"\n=== {name} ===")
    clf.fit(X_train_imputed, y_train)
    y_pred = clf.predict(X_test_imputed)
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()
    fpr = fp / (fp + tn)
    fpr_baseline[name]     = fpr
    baseline_results[name] = {"confusion_matrix": cm, "fpr": fpr}
    print(cm)
    print(f"Baseline FPR: {fpr:.4f}")
    print(classification_report(y_test, y_pred, digits=4))

clf_baseline    = models_baseline["RandomForest"]
fpr_baseline_rf = fpr_baseline["RandomForest"]



=== RandomForest ===
[[257805     29]
 [   407   1094]]
Baseline FPR: 0.0001
              precision    recall  f1-score   support

           0     0.9984    0.9999    0.9992    257834
           1     0.9742    0.7288    0.8338      1501

    accuracy                         0.9983    259335
   macro avg     0.9863    0.8644    0.9165    259335
weighted avg     0.9983    0.9983    0.9982    259335


=== GradientBoosting ===
[[257716    118]
 [   434   1067]]
Baseline FPR: 0.0005
              precision    recall  f1-score   support

           0     0.9983    0.9995    0.9989    257834
           1     0.9004    0.7109    0.7945      1501

    accuracy                         0.9979    259335
   macro avg     0.9494    0.8552    0.8967    259335
weighted avg     0.9978    0.9979    0.9977    259335


=== LogisticRegression ===
[[244601  13233]
 [   353   1148]]
Baseline FPR: 0.0513
              precision    recall  f1-score   support

           0     0.9986    0.9487    0.9730    

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [6]:
# =============================================================================
# CELL 7 — LLM TRAINING DATA PREPARATION (for fraud generator)
# =============================================================================
def tx_to_text(row):
    return (
        "{"
        f" 'trans_date_trans_time': '{row['trans_date_trans_time']}',"
        f" 'cc_num': '{int(row['cc_num'])}',"
        f" 'merchant': '{row['merchant']}',"
        f" 'category': '{row['category']}',"
        f" 'amt': {row['amt']:.2f},"
        f" 'city': '{row['city']}', 'state': '{row['state']}',"
        f" 'zip': '{row['zip']}', 'lat': {row['lat']}, 'long': {row['long']},"
        f" 'city_pop': {row['city_pop']}, 'job': '{row['job']}',"
        f" 'dob': '{row['dob']}', 'unix_time': {row['unix_time']},"
        f" 'merch_lat': {row['merch_lat']}, 'merch_long': {row['merch_long']},"
        f" 'merch_zipcode': '{row['merch_zipcode']}', 'is_fraud': 1"
        " }"
    )

_FRAUD_PROMPT_TEMPLATE = (
    "You are a fraud data generator. Given the following user profile, "
    "generate ONE realistic fraudulent credit card transaction for this user. "
    "Return ONLY a Python-style dictionary with keys matching the dataset, "
    "and set 'is_fraud' to 1.\n\n"
    "USER PROFILE:\n{profile_text}\n\nFRAUD TRANSACTION:\n"
)

df_fraud = df[df['is_fraud'] == 1].copy()
df_fraud = df_fraud.merge(
    user_profiles[['cc_num','profile_text']], on='cc_num', how='left')

prompts, completions = [], []
for _, row in df_fraud.iterrows():
    prompts.append(_FRAUD_PROMPT_TEMPLATE.format(profile_text=row['profile_text']))
    completions.append(tx_to_text(row))

hf_ds = Dataset.from_pandas(pd.DataFrame({"prompt": prompts, "completion": completions}))
hf_ds = hf_ds.map(lambda ex: {"text": ex["prompt"] + ex["completion"]},
                   remove_columns=['prompt','completion'])
print(f"LLM training examples: {len(hf_ds):,}")


Map:   0%|          | 0/7506 [00:00<?, ? examples/s]

LLM training examples: 7,506


In [7]:
# =============================================================================
# CELL 8 — FRAUD LLM  (Qwen2.5-0.5B · 4-bit QLoRA)
# Train-once: first run downloads + trains (~10 min) then saves to Drive.
# Every run after: loads from Drive in ~1 min. No phi-2, no version errors.
# =============================================================================
if not torch.cuda.is_available():
    raise EnvironmentError(
        "No GPU. Go to Runtime > Change runtime type > T4 GPU, then reconnect.")
print(f"GPU: {torch.cuda.get_device_name(0)}")

_FRAUD_MODEL_NAME   = "Qwen/Qwen2.5-0.5B-Instruct"
_FRAUD_MODEL_DIR    = "./fraud-llm"
_FRAUD_DRIVE_BACKUP = "/content/drive/MyDrive/Capstone/fraud-llm"

_bnb_fraud = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)

_fraud_src = (
    _FRAUD_MODEL_DIR    if os.path.exists(_FRAUD_MODEL_DIR)    else
    _FRAUD_DRIVE_BACKUP if os.path.exists(_FRAUD_DRIVE_BACKUP) else None)

if _fraud_src:
    print(f"Loading saved fraud-llm from '{_fraud_src}' ...")
    if _fraud_src == _FRAUD_DRIVE_BACKUP and not os.path.exists(_FRAUD_MODEL_DIR):
        print("  Copying from Drive ...")
        os.system(f'cp -r "{_FRAUD_DRIVE_BACKUP}" "{_FRAUD_MODEL_DIR}"')

    fraud_infer_tokenizer = AutoTokenizer.from_pretrained(_FRAUD_MODEL_DIR)
    if fraud_infer_tokenizer.pad_token is None:
        fraud_infer_tokenizer.pad_token = fraud_infer_tokenizer.eos_token

    _base = AutoModelForCausalLM.from_pretrained(
        _FRAUD_MODEL_NAME, quantization_config=_bnb_fraud,
        device_map="auto", trust_remote_code=True)
    fraud_gen_model = PeftModel.from_pretrained(_base, _FRAUD_MODEL_DIR)
    fraud_gen_model.eval()
    print("Fraud generator ready.")

else:
    print("No saved model — training from scratch ...")

    fraud_infer_tokenizer = AutoTokenizer.from_pretrained(_FRAUD_MODEL_NAME)
    if fraud_infer_tokenizer.pad_token is None:
        fraud_infer_tokenizer.pad_token = fraud_infer_tokenizer.eos_token

    _base_train = AutoModelForCausalLM.from_pretrained(
        _FRAUD_MODEL_NAME, quantization_config=_bnb_fraud,
        device_map="auto", trust_remote_code=True)
    _base_train = prepare_model_for_kbit_training(_base_train)
    fraud_gen_model = get_peft_model(_base_train, LoraConfig(
        r=16, lora_alpha=32, lora_dropout=0.05,
        target_modules="all-linear", bias="none", task_type="CAUSAL_LM"))
    fraud_gen_model.print_trainable_parameters()

    def _tok_fraud(ex):
        return fraud_infer_tokenizer(ex["text"], truncation=True, max_length=512)

    _tok_ds = hf_ds.map(_tok_fraud, batched=True, remove_columns=["text"])

    Trainer(
        model=fraud_gen_model,
        args=TrainingArguments(
            output_dir=_FRAUD_MODEL_DIR,
            per_device_train_batch_size=4, gradient_accumulation_steps=2,
            num_train_epochs=1, learning_rate=2e-4, fp16=True,
            logging_steps=50, save_total_limit=2, save_steps=500,
            optim="paged_adamw_8bit", dataloader_num_workers=2,
        ),
        train_dataset=_tok_ds,
        data_collator=DataCollatorForLanguageModeling(
            tokenizer=fraud_infer_tokenizer, mlm=False),
    ).train()

    fraud_gen_model.save_pretrained(_FRAUD_MODEL_DIR)
    fraud_infer_tokenizer.save_pretrained(_FRAUD_MODEL_DIR)
    os.makedirs(os.path.dirname(_FRAUD_DRIVE_BACKUP), exist_ok=True)
    os.system(f'cp -r "{_FRAUD_MODEL_DIR}" "{_FRAUD_DRIVE_BACKUP}"')
    print(f"Saved to {_FRAUD_DRIVE_BACKUP}")
    fraud_gen_model.eval()


GPU: Tesla T4
Loading saved fraud-llm from '/content/drive/MyDrive/Capstone/fraud-llm' ...
  Copying from Drive ...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Fraud generator ready.


In [8]:
# =============================================================================
# CELL 9 — SYNTHETIC FRAUD DATA GENERATION
# =============================================================================
import re

def _extract_dict_str(raw):
    if "FRAUD TRANSACTION:" in raw:
        raw = raw.split("FRAUD TRANSACTION:")[-1]
    for stop in ["\nA:", "\n\nA:", "\nAnswer:", "\nNote:", "\n\n\n"]:
        if stop in raw:
            raw = raw.split(stop)[0]

    # Try complete dict first
    match = re.search(r"\{.*?\}", raw, re.DOTALL)
    if not match:
        match = re.search(r"\{.*\}", raw, re.DOTALL)

    # ── KEY FIX: if no closing brace, the dict was truncated ─────────────────
    # Take everything from the opening brace and close it ourselves
    if not match:
        brace_start = raw.find("{")
        if brace_start != -1:
            c = raw[brace_start:]
            # Remove the last incomplete key-value pair (ends with comma or nothing)
            c = re.sub(r",\s*'[^']*'\s*:\s*[^,}]*$", "", c.rstrip())
            c = re.sub(r",\s*$", "", c)
            c = c + "}"
            match_candidate = c
        else:
            return ""
    else:
        match_candidate = match.group(0)

    c = match_candidate
    c = re.sub(r":\s*0+(\d+)(?=[,\}])", r': "\1"', c)
    while '"""' in c:
        c = c.replace('"""', '"')
    c = re.sub(r",\s*\}", "}", c)
    c = re.sub(r"#[^\n]*", "", c)
    if c.count("{") > c.count("}"):
        c += "}"
    return c.strip()


def _gen_single(prompt, max_new_tokens=300):
    fraud_infer_tokenizer.padding_side = "left"
    enc = fraud_infer_tokenizer(
        [prompt], return_tensors="pt", truncation=True, max_length=400
    ).to(fraud_gen_model.device)
    with torch.no_grad():
        out = fraud_gen_model.generate(
            **enc, max_new_tokens=max_new_tokens,
            do_sample=True, temperature=0.7, top_p=0.9,
            pad_token_id=fraud_infer_tokenizer.pad_token_id)
    return _extract_dict_str(
        fraud_infer_tokenizer.decode(out[0], skip_special_tokens=True))


def generate_fraud_batch(profile_texts, max_new_tokens=300, max_retries=3):
    prompts = [_FRAUD_PROMPT_TEMPLATE.format(profile_text=p) for p in profile_texts]
    fraud_infer_tokenizer.padding_side = "left"
    enc = fraud_infer_tokenizer(
        prompts, return_tensors="pt", padding=True,
        truncation=True, max_length=400
    ).to(fraud_gen_model.device)
    with torch.no_grad():
        outs = fraud_gen_model.generate(
            **enc, max_new_tokens=max_new_tokens,
            do_sample=True, temperature=0.7, top_p=0.9,
            pad_token_id=fraud_infer_tokenizer.pad_token_id)
    results = [
        _extract_dict_str(fraud_infer_tokenizer.decode(o, skip_special_tokens=True))
        for o in outs]

    for idx, (result, prompt) in enumerate(zip(results, prompts)):
        if result:
            continue
        for attempt in range(max_retries):
            r = _gen_single(prompt, max_new_tokens)
            if r:
                results[idx] = r
                break
            if attempt == max_retries - 2:
                enc2 = fraud_infer_tokenizer(
                    [prompt], return_tensors="pt",
                    truncation=True, max_length=400
                ).to(fraud_gen_model.device)
                with torch.no_grad():
                    o2 = fraud_gen_model.generate(
                        **enc2, max_new_tokens=max_new_tokens,
                        do_sample=True, temperature=0.95, top_p=0.95,
                        pad_token_id=fraud_infer_tokenizer.pad_token_id)
                r = _extract_dict_str(
                    fraud_infer_tokenizer.decode(o2[0], skip_special_tokens=True))
                if r:
                    results[idx] = r
                    break
    return results


BATCH_SIZE  = 8
SAMPLE_FRAC = 0.05

subset_profiles = user_profiles.sample(frac=SAMPLE_FRAC, random_state=42)
profile_list    = subset_profiles[['cc_num','profile_text']].values.tolist()
synthetic_rows, parse_ok, parse_fail = [], 0, 0

for batch_start in range(0, len(profile_list), BATCH_SIZE):
    batch   = profile_list[batch_start: batch_start + BATCH_SIZE]
    cc_nums = [r[0] for r in batch]
    texts   = [r[1] for r in batch]
    for cc_num, dict_str in zip(cc_nums, generate_fraud_batch(texts)):
        if not dict_str:
            parse_fail += 1
            continue
        try:
            tx = ast.literal_eval(dict_str)
            tx['cc_num']   = cc_num
            tx['is_fraud'] = 1
            synthetic_rows.append(tx)
            parse_ok += 1
        except Exception:
            parse_fail += 1
    done = min(batch_start + BATCH_SIZE, len(profile_list))
    print(f"  {done}/{len(profile_list)} — {parse_ok} ok  {parse_fail} failed", end="\r")

total = parse_ok + parse_fail
print(f"\nParsed: {parse_ok}  Failed: {parse_fail}  "
      f"Yield: {parse_ok/total:.0%}" if total else "\nNo output.")

df_synth = pd.DataFrame(synthetic_rows)
for col in df.columns:
    if col not in df_synth.columns:
        df_synth[col] = np.nan
df_synth = df_synth[df.columns]
print(f"df_synth shape: {df_synth.shape}")
df_synth.head()

  45/45 — 43 ok  2 failed
Parsed: 43  Failed: 2  Yield: 96%
df_synth shape: (43, 27)


,Unnamed: 0,trans_date_trans_time,cc_num,merchant,category,amt,first,last,gender,street,...,dob,trans_num,unix_time,merch_lat,merch_long,is_fraud,merch_zipcode,age,tx_hour,tx_dayofweek
0,NaN,2020-01-07 03:25:04,4477156602511939689,fraud_Rempel Inc,grocery_pos,309.87,NaN,NaN,NaN,NaN,...,1945-11-14 00:00:00,NaN,1357858704,41.200921,-112.943352,1,nan,NaN,NaN,NaN
1,NaN,2019-08-27 23:00:27,3512365128314616,fraud_Beier-Hyatt,shopping_net,1159.67,NaN,NaN,NaN,NaN,...,1970-05-14 00:00:00,NaN,1346654827,39.199263,-90.610477,1,63019.0,NaN,NaN,NaN
2,NaN,2019-09-01 23:07:44,375248307838179,"fraud_Gottlieb, Considine and Schultz",shopping_net,940.98,NaN,NaN,NaN,NaN,...,1959-04-07 00:00:00,NaN,1347857064,38.671545,-81.724196,1,40833.0,NaN,NaN,NaN
3,NaN,2019-12-28 23:35:42,5127412150261034,fraud_Schmeler Inc,grocery_pos,289.61,NaN,NaN,NaN,NaN,...,1999-07-17 00:00:00,NaN,1356924942,38.903337,-76.778161,1,20451.0,NaN,NaN,NaN
4,NaN,2020-01-13 03:38:18,4208110975550360171,fraud_Cormier LLC,grocery_pos,339.66,NaN,NaN,NaN,NaN,...,1978-02-14 00:00:00,NaN,1357994798,31.618991,-81.885637,1,nan,NaN,NaN,NaN


In [9]:
# =============================================================================
# CELL 10 — AUGMENTED MODEL TRAINING
# =============================================================================

if df_synth.empty:
    print("⚠️  df_synth is empty — skipping augmented training.")
    print("   Fix Cell 9 yield rate first, then re-run this cell.")
else:
    df_synth_enc = df_synth.copy()

    # Fill missing cardholder fields from original data.
    # Drop ALL personal cols from df_synth first (whether NaN or LLM-generated)
    # so the merge with _lookup never creates duplicate columns.
    _personal = ['first','last','gender','street','city','state',
                 'zip','lat','long','city_pop','job','dob']
    _lookup = (df.sort_values('trans_date_trans_time')
                 .groupby('cc_num').last()[_personal].reset_index())

    _cols_to_drop = [c for c in _personal if c in df_synth_enc.columns]
    df_synth_enc = df_synth_enc.drop(columns=_cols_to_drop)
    df_synth_enc = df_synth_enc.merge(_lookup, on='cc_num', how='left')

    df_synth_enc['trans_date_trans_time'] = pd.to_datetime(
        df_synth_enc['trans_date_trans_time'], errors='coerce')
    df_synth_enc['dob'] = pd.to_datetime(df_synth_enc['dob'], errors='coerce')
    df_synth_enc['age']          = (df_synth_enc['trans_date_trans_time'].dt.year
                                    - df_synth_enc['dob'].dt.year)
    df_synth_enc['tx_hour']      = df_synth_enc['trans_date_trans_time'].dt.hour
    df_synth_enc['tx_dayofweek'] = df_synth_enc['trans_date_trans_time'].dt.dayofweek
    df_synth_enc = df_synth_enc.merge(
        user_profiles, on='cc_num', how='left', suffixes=('','_prof'))

    for col in feature_cols_cat:
        df_synth_enc[col] = df_synth_enc[col].fillna('UNKNOWN_CATEGORY').astype(str)
        mask = ~df_synth_enc[col].isin(label_encoders[col].classes_)
        df_synth_enc.loc[mask, col] = 'UNKNOWN_CATEGORY'
        df_synth_enc[col] = label_encoders[col].transform(df_synth_enc[col])

    X_synth = df_synth_enc[feature_cols_num + feature_cols_cat]
    y_synth = df_synth_enc['is_fraud'].astype(int)
    X_train_aug = pd.concat([X_train, X_synth])
    y_train_aug = pd.concat([y_train, y_synth])
    X_train_aug_imputed = pd.DataFrame(
        imputer.transform(X_train_aug),
        columns=X_train_aug.columns, index=X_train_aug.index)

    models_augmented = {
        "RandomForest"      : RandomForestClassifier(
            n_estimators=200, n_jobs=-1, random_state=42, class_weight='balanced'),
        "GradientBoosting"  : GradientBoostingClassifier(random_state=42),
        "LogisticRegression": LogisticRegression(max_iter=1000, class_weight='balanced'),
    }
    augmented_results, fpr_augmented = {}, {}

    for name, clf in models_augmented.items():
        print(f"\n=== {name} — augmented ===")
        clf.fit(X_train_aug_imputed, y_train_aug)
        y_pred = clf.predict(X_test_imputed)
        cm = confusion_matrix(y_test, y_pred)
        tn, fp, fn, tp = cm.ravel()
        fpr_a = fp / (fp + tn)
        fpr_augmented[name]     = fpr_a
        augmented_results[name] = {"confusion_matrix": cm, "fpr": fpr_a}
        print(cm)
        print(f"FPR: {fpr_a:.4f}  baseline: {fpr_baseline.get(name,0):.4f}  "
              f"delta: {fpr_a - fpr_baseline.get(name,0):+.4f}")
        print(classification_report(y_test, y_pred, digits=4))


=== RandomForest — augmented ===
[[257808     26]
 [   409   1092]]
FPR: 0.0001  baseline: 0.0001  delta: -0.0000
              precision    recall  f1-score   support

           0     0.9984    0.9999    0.9992    257834
           1     0.9767    0.7275    0.8339      1501

    accuracy                         0.9983    259335
   macro avg     0.9876    0.8637    0.9165    259335
weighted avg     0.9983    0.9983    0.9982    259335


=== GradientBoosting — augmented ===
[[257606    228]
 [   540    961]]
FPR: 0.0009  baseline: 0.0005  delta: +0.0004
              precision    recall  f1-score   support

           0     0.9979    0.9991    0.9985    257834
           1     0.8082    0.6402    0.7145      1501

    accuracy                         0.9970    259335
   macro avg     0.9031    0.8197    0.8565    259335
weighted avg     0.9968    0.9970    0.9969    259335


=== LogisticRegression — augmented ===
[[243627  14207]
 [   357   1144]]
FPR: 0.0551  baseline: 0.0513  delta:

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [10]:
# =============================================================================
# CELL 11 — DUCKDB SETUP
# =============================================================================
_GDRIVE_FILE_ID = '1nON5KbGGszgy5wsPoVtLVGcI5sbh51pq'
_LOCAL_CSV      = 'credit_card_data.csv'
_DUCKDB_FILE    = 'transactions.duckdb'

if not os.path.exists(_LOCAL_CSV):
    print('Downloading CSV ...')
    os.system(f'gdown --id {_GDRIVE_FILE_ID} -O {_LOCAL_CSV}')

_con = None

def get_con():
    """Always returns a live DuckDB connection. Auto-reconnects if closed."""
    global _con, _DUCKDB_FILE
    try:
        if _con is None:
            raise Exception("not connected")
        _con.execute("SELECT 1")
    except Exception:
        _con = duckdb.connect(_DUCKDB_FILE)
    return _con

_con = duckdb.connect(_DUCKDB_FILE)
_con.execute('DROP TABLE IF EXISTS transactions')
_con.execute(f"CREATE TABLE transactions AS SELECT * FROM read_csv_auto('{_LOCAL_CSV}')")

print('DuckDB ready. Summary:')
print(get_con().execute("""
    SELECT COUNT(*) AS rows,
           COUNT(DISTINCT cc_num) AS cards,
           COUNT(DISTINCT category) AS categories,
           MIN(trans_date_trans_time) AS first_tx,
           MAX(trans_date_trans_time) AS last_tx
    FROM transactions
""").fetchdf().to_string(index=False))

df_sample = get_con().execute("""
    SELECT cc_num,
           MIN(trans_date_trans_time) AS min_dt,
           MAX(trans_date_trans_time) AS max_dt
    FROM transactions
    GROUP BY cc_num
    LIMIT 1000
""").fetchdf()
print(f"Cards sampled for NL->SQL training: {len(df_sample)}")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

DuckDB ready. Summary:
   rows  cards  categories            first_tx             last_tx
1296675    983          14 2019-01-01 00:00:18 2020-06-21 12:13:37
Cards sampled for NL->SQL training: 983


In [11]:
# =============================================================================
# CELL 12 — NL->SQL TRAINING DATA GENERATION
# =============================================================================
def random_date_range(min_dt, max_dt):
    a, b = pd.to_datetime(min_dt), pd.to_datetime(max_dt)
    if a >= b:
        return a.date().isoformat(), b.date().isoformat()
    t1 = a + (b - a) * random.random()
    t2 = a + (b - a) * random.random()
    s, e = sorted([t1, t2])
    return s.date().isoformat(), e.date().isoformat()

_SCHEMA_TEXT = """
Table transactions(
    row_index INT, trans_date_trans_time TIMESTAMP, cc_num BIGINT,
    merchant TEXT, category TEXT, amt DOUBLE,
    first TEXT, last TEXT, gender TEXT, street TEXT,
    city TEXT, state TEXT, zip INT, lat DOUBLE, long DOUBLE,
    city_pop INT, job TEXT, dob DATE, trans_num TEXT,
    unix_time BIGINT, merch_lat DOUBLE, merch_long DOUBLE,
    is_fraud INT, merch_zipcode INT
)"""

records = []
for _, row in df_sample.iterrows():
    card = int(row["cc_num"])
    s, e = random_date_range(row["min_dt"], row["max_dt"])
    records.append({
        "question": f"What is the total amount spent by card {card} between {s} and {e}?",
        "sql": (f"SELECT SUM(amt) AS total_amount FROM transactions "
                f"WHERE cc_num = {card} "
                f"AND trans_date_trans_time >= '{s}' "
                f"AND trans_date_trans_time < '{e}';")
    })
    records.append({
        "question": f"Which merchants has card {card} spent the most money at?",
        "sql": (f"SELECT merchant, SUM(amt) AS total_spent FROM transactions "
                f"WHERE cc_num = {card} "
                f"GROUP BY merchant ORDER BY total_spent DESC LIMIT 10;")
    })
for _ in range(300):
    records.append({
        "question": "What is the fraud rate by transaction category?",
        "sql": ("SELECT category, AVG(is_fraud::DOUBLE) AS fraud_rate, "
                "COUNT(*) AS total_txn FROM transactions "
                "GROUP BY category ORDER BY fraud_rate DESC;")
    })

def _fmt(ex):
    return {"text": (
        "### Instruction:\n"
        "You are an assistant that writes SQL queries for DuckDB.\n"
        "Only output a single SQL query, nothing else.\n\n"
        f"### Schema:\n{_SCHEMA_TEXT}\n\n"
        f"### Question:\n{ex['question']}\n\n"
        f"### Response (SQL only):\n{ex['sql']}"
    )}

ds_train = (Dataset.from_list(records)
            .map(_fmt)
            .remove_columns(['question','sql'])
            .shuffle(seed=42))
print(f"NL->SQL training records: {len(ds_train)}")


Map:   0%|          | 0/2266 [00:00<?, ? examples/s]

NL->SQL training records: 2266


In [12]:
# =============================================================================
# CELL 13 — NL->SQL LLM  (Qwen2.5-0.5B · 4-bit QLoRA)
# Train-once: first run trains (~10 min) and saves to Drive.
# Every run after: loads from Drive in ~1 min.
# =============================================================================
_NL2SQL_MODEL_NAME   = "Qwen/Qwen2.5-0.5B-Instruct"
_NL2SQL_MODEL_DIR    = "./nl2sql-cc-finetuned"
_NL2SQL_DRIVE_BACKUP = "/content/drive/MyDrive/Capstone/nl2sql-cc-finetuned"

_bnb_nl = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)

_nl2sql_src = (
    _NL2SQL_MODEL_DIR    if os.path.exists(_NL2SQL_MODEL_DIR)    else
    _NL2SQL_DRIVE_BACKUP if os.path.exists(_NL2SQL_DRIVE_BACKUP) else None)

if _nl2sql_src:
    print(f"Loading NL->SQL adapter from '{_nl2sql_src}' ...")
    if _nl2sql_src == _NL2SQL_DRIVE_BACKUP and not os.path.exists(_NL2SQL_MODEL_DIR):
        print("  Copying from Drive ...")
        os.system(f'cp -r "{_NL2SQL_DRIVE_BACKUP}" "{_NL2SQL_MODEL_DIR}"')

    tokenizer = AutoTokenizer.from_pretrained(_NL2SQL_MODEL_DIR)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    _nl_base = AutoModelForCausalLM.from_pretrained(
        _NL2SQL_MODEL_NAME, quantization_config=_bnb_nl,
        device_map="auto", trust_remote_code=True)
    model = PeftModel.from_pretrained(_nl_base, _NL2SQL_MODEL_DIR)
    model.eval()
    print("NL->SQL model ready.")

else:
    print("No saved adapter — training from scratch ...")

    tokenizer = AutoTokenizer.from_pretrained(_NL2SQL_MODEL_NAME)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        _NL2SQL_MODEL_NAME, quantization_config=_bnb_nl,
        device_map="auto", trust_remote_code=True)
    model = prepare_model_for_kbit_training(model)
    model = get_peft_model(model, LoraConfig(
        r=16, lora_alpha=32, lora_dropout=0.05,
        target_modules=["q_proj","v_proj","k_proj","o_proj"],
        bias="none", task_type="CAUSAL_LM"))
    model.print_trainable_parameters()

    def _tok_nl(ex):
        t = tokenizer(ex["text"], truncation=True, max_length=512, padding="max_length")
        t["labels"] = t["input_ids"].copy()
        return t

    _ds_tok = ds_train.map(_tok_nl, batched=True, remove_columns=["text"])
    _ds_tok = _ds_tok.train_test_split(test_size=0.05, seed=42)

    Trainer(
        model=model,
        args=TrainingArguments(
            output_dir=_NL2SQL_MODEL_DIR,
            per_device_train_batch_size=2, per_device_eval_batch_size=2,
            gradient_accumulation_steps=4, num_train_epochs=0.5,
            learning_rate=2e-4, warmup_steps=50, fp16=True,
            logging_steps=50, save_steps=500, eval_steps=500,
            report_to="none", lr_scheduler_type="cosine",
        ),
        train_dataset=_ds_tok["train"],
        eval_dataset=_ds_tok["test"],
    ).train()

    model.save_pretrained(_NL2SQL_MODEL_DIR)
    tokenizer.save_pretrained(_NL2SQL_MODEL_DIR)
    os.makedirs(os.path.dirname(_NL2SQL_DRIVE_BACKUP), exist_ok=True)
    os.system(f'cp -r "{_NL2SQL_MODEL_DIR}" "{_NL2SQL_DRIVE_BACKUP}"')
    print(f"Saved to {_NL2SQL_DRIVE_BACKUP}")
    model.eval()


Loading NL->SQL adapter from '/content/drive/MyDrive/Capstone/nl2sql-cc-finetuned' ...
  Copying from Drive ...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

NL->SQL model ready.


In [13]:
# =============================================================================
# CELL 14 — INFERENCE + VISUALIZATION FUNCTIONS
# =============================================================================

def build_prompt(question):
    return (
        "### Instruction:\n"
        "You are an assistant that writes SQL queries for DuckDB.\n"
        "Only output a single SQL query, nothing else.\n\n"
        f"### Schema:\n{_SCHEMA_TEXT}\n\n"
        f"### Question:\n{question}\n\n"
        "### Response (SQL only):"
    )

def generate_sql(question, max_new_tokens=256):
    inputs = tokenizer(build_prompt(question), return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id)
    full = tokenizer.decode(out[0], skip_special_tokens=True)

    # Extract only the part after the response marker
    if "### Response (SQL only):" in full:
        raw = full.split("### Response (SQL only):")[-1]
    else:
        raw = full

    # Strip markdown code fences (```sql ... ``` or ``` ... ```)
    raw = re.sub(r"```(?:sql)?", "", raw, flags=re.IGNORECASE).strip()

    # Remove comment-only lines (-- ...)
    lines = [l for l in raw.splitlines() if not l.strip().startswith("--")]
    raw = "\n".join(lines).strip()

    # Try to extract the first complete SELECT statement
    match = re.search(r"(SELECT\b[\s\S]*?;)", raw, re.IGNORECASE)
    if match:
        sql = match.group(1).strip()
    else:
        # No semicolon — take everything up to a blank line or end
        sql = raw.split("\n\n")[0].strip()
        if not sql.upper().startswith("SELECT"):
            sql = ""

    return sql if sql else raw.split(";")[0].strip() + ";"

def choose_axes(df, question):
    if df.empty:
        return None, None, None
    cols  = list(df.columns)
    num   = [c for c in cols if is_numeric_dtype(df[c])]
    dts   = [c for c in cols if is_datetime64_any_dtype(df[c])]
    cat   = [c for c in cols if df[c].dtype == "object"]
    q     = question.lower()
    tq    = any(k in q for k in ["over time","by day","by month","trend","daily","monthly"])
    if tq and not dts:
        for c in cols:
            if "date" in c.lower() or "time" in c.lower():
                try:
                    df[c] = pd.to_datetime(df[c])
                    dts.append(c)
                except Exception:
                    pass
    if tq and dts and num:
        return dts[0], next((c for c in num if "id" not in c.lower()), num[0]), "line"
    if cat and num:
        return cat[0], next((c for c in num if c != cat[0]), num[0]), "bar"
    if len(num) >= 2:
        return num[0], num[1], "scatter"
    return None, None, None

def create_plot(df, question):
    if df is None or df.empty:
        return None
    x, y, kind = choose_axes(df, question)
    if not all([x, y, kind]):
        return None
    dp = df[[x, y]].dropna()
    if dp.empty:
        return None
    if len(dp) > 500:
        dp = dp.sample(500, random_state=42)
    fig, ax = plt.subplots()
    if kind == "line":
        dp = dp.sort_values(x)
        ax.plot(dp[x], dp[y])
    elif kind == "bar":
        if dp[x].nunique() > 20:
            dp = (dp.groupby(x, as_index=False)[y]
                    .sum()
                    .sort_values(y, ascending=False)
                    .head(20))
        ax.bar(dp[x], dp[y])
        ax.set_xticklabels(dp[x], rotation=45, ha="right")
    elif kind == "scatter":
        ax.scatter(dp[x], dp[y])
    ax.set_xlabel(x)
    ax.set_ylabel(y)
    ax.set_title(question)
    fig.tight_layout()
    return fig

def query_agent(question):
    """Run NL -> SQL -> DuckDB and return (sql, DataFrame). Simple version without viz."""
    sql = generate_sql(question)
    print("Generated SQL:", sql)
    try:
        con = get_con()
        if con is None:
            raise RuntimeError("DuckDB not initialised — run Cell 11 first.")
        df_result = con.execute(sql).fetchdf()
    except Exception as e:
        print("SQL execution error:", e)
        df_result = pd.DataFrame()
    return sql, df_result

def query_agent_with_viz(question):
    """NL -> SQL -> DuckDB -> (sql, DataFrame, Figure, error_message)."""
    if not question.strip():
        return "", pd.DataFrame(), None, "Please enter a question."
    try:
        sql = generate_sql(question)
    except Exception as e:
        return "", pd.DataFrame(), None, f"SQL generation error: {e}"
    if not sql.strip().upper().lstrip().startswith("SELECT"):
        return sql, pd.DataFrame(), None, f"Only SELECT queries are allowed. Got: {sql[:80]!r}"
    try:
        con = get_con()
        if con is None:
            raise RuntimeError("DuckDB not initialised — run Cell 11 first.")
        df_result = con.execute(sql).fetchdf()
    except Exception as e:
        return sql, pd.DataFrame(), None, f"SQL execution error: {e}"
    try:
        fig = create_plot(df_result, question)
    except Exception as e:
        return sql, df_result, None, f"Query ran; visualization failed: {e}"
    return sql, df_result, fig, ""

print("Inference and visualization functions defined.")


Inference and visualization functions defined.


In [14]:
import re
from difflib import get_close_matches

VALID_COLUMNS = [
    "row_index", "trans_date_trans_time", "cc_num", "merchant", "category",
    "amt", "first", "last", "gender", "street", "city", "state", "zip",
    "lat", "long", "city_pop", "job", "dob", "trans_num", "unix_time",
    "merch_lat", "merch_long", "is_fraud", "merch_zipcode"
]

SQL_KEYWORDS = {
    'select','from','where','group','by','order','having','limit','as',
    'and','or','not','in','is','null','on','join','inner','left','right',
    'outer','union','all','distinct','case','when','then','else','end',
    'sum','count','avg','min','max','coalesce','cast','between','like',
    'asc','desc','with','exists','any','top','offset','fetch','next',
    'rows','only','over','partition','int','double','text','bigint',
    'timestamp','date','boolean','true','false','transactions','max','min'
}

def correct_column_names(sql):
    tokens = re.split(r'(\s+|,|\(|\)|;)', sql)
    result = []
    skip_next = False
    for token in tokens:
        stripped = token.strip()
        if skip_next:
            result.append(token)
            if stripped:
                skip_next = False
            continue
        if stripped.upper() == 'AS':
            skip_next = True
            result.append(token)
            continue
        if (not stripped or stripped.lower() in SQL_KEYWORDS
                or re.match(r'^[\d\.]+$', stripped)
                or stripped.startswith("'") or stripped.startswith('"')):
            result.append(token)
            continue
        if re.match(r'^[a-zA-Z_][a-zA-Z0-9_]*$', stripped):
            if stripped not in VALID_COLUMNS and stripped.lower() not in SQL_KEYWORDS:
                close = get_close_matches(stripped.lower(), VALID_COLUMNS, n=1, cutoff=0.75)
                if close:
                    token = token.replace(stripped, close[0])
        result.append(token)
    return ''.join(result)


def is_valid_sql(sql):
    """Use DuckDB's own parser to validate SQL without executing it."""
    try:
        get_con().execute(f"EXPLAIN {sql}")
        return True
    except Exception as e:
        return False


def extract_sql(raw):
    """Pull the first SELECT statement out of raw model output."""
    raw = re.sub(r"```(?:sql)?\s*", "", raw, flags=re.IGNORECASE).strip()
    lines = [l for l in raw.splitlines() if not l.strip().startswith("--")]
    raw = "\n".join(lines).strip()

    match = re.search(r"(SELECT\b[\s\S]*?;)", raw, re.IGNORECASE)
    if match:
        return match.group(1).strip()
    match2 = re.search(r"(SELECT\b[\s\S]*)", raw, re.IGNORECASE)
    if match2:
        return match2.group(1).strip().split("\n\n")[0].strip() + ";"
    return raw + ";"


def fix_sql(sql):
    """Fix column names, GROUP BY, and WHERE/HAVING issues."""
    sql = correct_column_names(sql)
    s = re.sub(r'\s+', ' ', sql).strip()
    if not s.upper().startswith("SELECT"):
        return sql

    # Move aggregates from WHERE to HAVING
    where_match = re.search(
        r'\bWHERE\b(.*?)(?=\bGROUP\b|\bORDER\b|\bLIMIT\b|;|$)',
        s, re.IGNORECASE)
    if where_match:
        where_body = where_match.group(1).strip()
        conditions = re.split(r'\b(AND|OR)\b', where_body, flags=re.IGNORECASE)
        where_conds, having_conds = [], []
        for cond in conditions:
            if cond.upper().strip() in ('AND', 'OR'):
                continue
            if re.search(r'\b(SUM|COUNT|AVG|MIN|MAX)\s*\(', cond, re.IGNORECASE):
                having_conds.append(cond.strip())
            else:
                where_conds.append(cond.strip())
        new_where  = f"WHERE {' AND '.join(where_conds)} " if where_conds else ""
        new_having = f"HAVING {' AND '.join(having_conds)} " if having_conds else ""
        s = re.sub(r'\bWHERE\b.*?(?=\bGROUP\b|\bORDER\b|\bLIMIT\b|;|$)',
                   new_where, s, flags=re.IGNORECASE)
        if new_having:
            s = re.sub(r'(?=\bORDER\s+BY\b|\bLIMIT\b|;|$)',
                       new_having, s, count=1, flags=re.IGNORECASE)

    # Fix missing GROUP BY
    outer_select = re.match(r'^SELECT\s+(.*?)\s+FROM\s+\w+', s, re.IGNORECASE)
    has_agg      = bool(re.search(r'\b(SUM|COUNT|AVG|MIN|MAX)\s*\(', s, re.IGNORECASE))
    has_group_by = bool(re.search(r'\bGROUP\s+BY\b', s, re.IGNORECASE))

    if outer_select and has_agg:
        select_part = outer_select.group(1)
        bare_cols = []
        for col in select_part.split(','):
            col = col.strip()
            if re.match(r'(SUM|COUNT|AVG|MIN|MAX)\s*\(', col, re.IGNORECASE):
                continue
            col_name = re.split(r'\s+AS\s+', col, flags=re.IGNORECASE)[0].strip()
            if col_name and re.match(r'^[a-zA-Z_][a-zA-Z0-9_]*$', col_name):
                bare_cols.append(col_name)

        if bare_cols:
            if has_group_by:
                gb_match = re.search(r'\bGROUP\s+BY\s+(.*?)(?=\bORDER\b|\bHAVING\b|\bLIMIT\b|;|$)', s, re.IGNORECASE)
                if gb_match:
                    existing = [g.strip() for g in gb_match.group(1).split(',')]
                    missing  = [c for c in bare_cols if c.lower() not in [e.lower() for e in existing]]
                    if missing:
                        new_gb = ', '.join(existing + missing)
                        s = re.sub(
                            r'\bGROUP\s+BY\s+.*?(?=\bORDER\b|\bHAVING\b|\bLIMIT\b|;|$)',
                            f'GROUP BY {new_gb} ', s, flags=re.IGNORECASE)
            else:
                s = re.sub(
                    r'(?=\bORDER\s+BY\b|\bLIMIT\b|;|$)',
                    f' GROUP BY {", ".join(bare_cols)} ',
                    s, count=1, flags=re.IGNORECASE)

    return s.strip().rstrip(';') + ';'


def generate_sql(question, max_new_tokens=256, max_retries=3):
    prompt = build_prompt(question)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    input_len = inputs["input_ids"].shape[1]

    for attempt in range(max_retries):
        # Slightly raise temperature on retries to get different output
        temp = 0.1 + (attempt * 0.2)

        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                min_new_tokens=10,
                do_sample=True,
                temperature=temp,
                top_p=0.95,
                repetition_penalty=1.1,
                pad_token_id=tokenizer.eos_token_id,
            )

        new_tokens = out[0][input_len:]
        raw = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
        sql = extract_sql(raw)
        sql = fix_sql(sql)

        print(f"Attempt {attempt+1} (temp={temp:.1f}): {sql[:120]}")

        if is_valid_sql(sql):
            print(f"✅ Valid SQL on attempt {attempt+1}")
            return sql
        else:
            print(f"❌ Invalid SQL, retrying...")

    # All retries failed — return last attempt anyway
    print("⚠️ All retries failed, returning best effort")
    return sql

print("✅ generate_sql with retry logic loaded — test with:")
print('   generate_sql("merchant with most fraudulent case")')

✅ generate_sql with retry logic loaded — test with:
   generate_sql("merchant with most fraudulent case")


In [15]:
def is_valid_sql(sql):
    """Use LIMIT 0 instead of EXPLAIN — actually catches binder/column errors."""
    try:
        # Strip trailing semicolon, wrap with LIMIT 0 to validate without full scan
        test_sql = sql.rstrip(';').strip()
        test_sql = f"SELECT * FROM ({test_sql}) __validation_query__ LIMIT 0;"
        get_con().execute(test_sql)
        return True
    except Exception as e:
        print(f"   Validation error: {e}")
        return False

print("✅ is_valid_sql fixed — now properly catches column/binder errors")

✅ is_valid_sql fixed — now properly catches column/binder errors


In [16]:
# Explicit mappings for columns the model consistently hallucinates
COLUMN_ALIASES = {
    "fraud_amount": "amt",
    "amount":       "amt",
    "transaction_amount": "amt",
    "total_amount": "amt",
    "card":         "cc_num",
    "card_num":     "cc_num",
    "card_number":  "cc_num",
    "credit_card":  "cc_num",
    "fraud":        "is_fraud",
    "is_fraudulent":"is_fraud",
    "fraudulent":   "is_fraud",
    "transaction_date": "trans_date_trans_time",
    "transaction_date_trans_time": "trans_date_trans_time",
    "date":         "trans_date_trans_time",
    "transaction_id": "trans_num",
    "zipcode":      "merch_zipcode",
    "zip_code":     "zip",
    "latitude":     "lat",
    "longitude":    "long",
    "population":   "city_pop",
}

def correct_column_names(sql):
    tokens = re.split(r'(\s+|,|\(|\)|;)', sql)
    result = []
    skip_next = False
    for token in tokens:
        stripped = token.strip()
        if skip_next:
            result.append(token)
            if stripped:
                skip_next = False
            continue
        if stripped.upper() == 'AS':
            skip_next = True
            result.append(token)
            continue
        if (not stripped or stripped.lower() in SQL_KEYWORDS
                or re.match(r'^[\d\.]+$', stripped)
                or stripped.startswith("'") or stripped.startswith('"')):
            result.append(token)
            continue
        if re.match(r'^[a-zA-Z_][a-zA-Z0-9_]*$', stripped):
            lower = stripped.lower()
            if lower in COLUMN_ALIASES:
                # Exact alias match
                corrected = COLUMN_ALIASES[lower]
                print(f"DEBUG fix_columns: '{stripped}' → '{corrected}' (alias map)")
                token = token.replace(stripped, corrected)
            elif stripped not in VALID_COLUMNS and lower not in SQL_KEYWORDS:
                # Fuzzy fallback with lower cutoff
                close = get_close_matches(lower, VALID_COLUMNS, n=1, cutoff=0.6)
                if close:
                    print(f"DEBUG fix_columns: '{stripped}' → '{close[0]}' (fuzzy)")
                    token = token.replace(stripped, close[0])
        result.append(token)
    return ''.join(result)

print("✅ correct_column_names updated with explicit alias map")

✅ correct_column_names updated with explicit alias map


In [18]:
# =============================================================================
# CELL 16 — SAVE ALL ARTIFACTS TO GOOGLE DRIVE
# DuckDB connection is NOT closed — Gradio UI stays active after saving.
# =============================================================================
_SAVE_DIR = '/content/drive/MyDrive/Capstone'
os.makedirs(_SAVE_DIR, exist_ok=True)

os.system(f'cp -r "{_FRAUD_MODEL_DIR}"  "{_SAVE_DIR}/"')
os.system(f'cp -r "{_NL2SQL_MODEL_DIR}" "{_SAVE_DIR}/"')
os.system(f'cp    "{_DUCKDB_FILE}"      "{_SAVE_DIR}/"')

print(f"All artifacts saved to {_SAVE_DIR}")
print("DuckDB connection kept open — Gradio UI remains active.")


All artifacts saved to /content/drive/MyDrive/Capstone
DuckDB connection kept open — Gradio UI remains active.


In [ ]:
# =============================================================================
# CELL 17 — STANDALONE: Load saved models + run Gradio (skip training cells)
# Run Cell 1 (install), Cell 2 (imports), Cell 3 (load data), Cell 11 (DuckDB)
# then jump straight to THIS cell to launch the UI with saved models.
# =============================================================================
import os, re, torch, duckdb, gradio as gr
import pandas as pd
import matplotlib.pyplot as plt
from difflib import get_close_matches
from pandas.api.types import is_numeric_dtype, is_datetime64_any_dtype
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

# ── Paths ──────────────────────────────────────────────────────────────────
_FRAUD_MODEL_NAME   = "Qwen/Qwen2.5-0.5B-Instruct"
_FRAUD_MODEL_DIR    = "./fraud-llm"
_FRAUD_DRIVE_BACKUP = "/content/drive/MyDrive/Capstone/fraud-llm"
_NL2SQL_MODEL_NAME  = "Qwen/Qwen2.5-0.5B-Instruct"
_NL2SQL_MODEL_DIR   = "./nl2sql-cc-finetuned"
_NL2SQL_DRIVE_BACKUP= "/content/drive/MyDrive/Capstone/nl2sql-cc-finetuned"
_DUCKDB_FILE        = "transactions.duckdb"
_GDRIVE_FILE_ID     = "1nON5KbGGszgy5wsPoVtLVGcI5sbh51pq"
_LOCAL_CSV          = "credit_card_data.csv"

# ── Load NL->SQL model ─────────────────────────────────────────────────────
_bnb = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)

_nl2sql_src = (_NL2SQL_MODEL_DIR    if os.path.exists(_NL2SQL_MODEL_DIR)    else
               _NL2SQL_DRIVE_BACKUP if os.path.exists(_NL2SQL_DRIVE_BACKUP) else None)

if not _nl2sql_src:
    raise FileNotFoundError("NL->SQL model not found. Run Cell 17 (train) first.")

if _nl2sql_src == _NL2SQL_DRIVE_BACKUP and not os.path.exists(_NL2SQL_MODEL_DIR):
    print("Copying NL->SQL model from Drive...")
    os.system(f'cp -r "{_NL2SQL_DRIVE_BACKUP}" "{_NL2SQL_MODEL_DIR}"')

print(f"Loading NL->SQL model from {_nl2sql_src}...")
tokenizer = AutoTokenizer.from_pretrained(_NL2SQL_MODEL_DIR)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
_nl_base = AutoModelForCausalLM.from_pretrained(
    _NL2SQL_MODEL_NAME, quantization_config=_bnb,
    device_map="auto", trust_remote_code=True)
model = PeftModel.from_pretrained(_nl_base, _NL2SQL_MODEL_DIR)
model.eval()
print("✅ NL->SQL model ready")

# ── DuckDB ─────────────────────────────────────────────────────────────────
_con = None
def get_con():
    global _con
    try:
        if _con is None: raise Exception()
        _con.execute("SELECT 1")
    except:
        if os.path.exists(_DUCKDB_FILE):
            _con = duckdb.connect(_DUCKDB_FILE)
        else:
            if not os.path.exists(_LOCAL_CSV):
                print("Downloading CSV...")
                os.system(f"gdown --id {_GDRIVE_FILE_ID} -O {_LOCAL_CSV}")
            _con = duckdb.connect(_DUCKDB_FILE)
            _con.execute("DROP TABLE IF EXISTS transactions")
            _con.execute(f"CREATE TABLE transactions AS SELECT * FROM read_csv_auto('{_LOCAL_CSV}')")
            print("✅ DuckDB ready")
    return _con

get_con()
print("✅ DuckDB connected")

# ── Schema ─────────────────────────────────────────────────────────────────
_SCHEMA_TEXT = """
Table transactions(
    row_index INT, trans_date_trans_time TIMESTAMP, cc_num BIGINT,
    merchant TEXT, category TEXT, amt DOUBLE,
    first TEXT, last TEXT, gender TEXT, street TEXT,
    city TEXT, state TEXT, zip INT, lat DOUBLE, long DOUBLE,
    city_pop INT, job TEXT, dob DATE, trans_num TEXT,
    unix_time BIGINT, merch_lat DOUBLE, merch_long DOUBLE,
    is_fraud INT, merch_zipcode INT
)"""

VALID_COLUMNS = [
    "row_index","trans_date_trans_time","cc_num","merchant","category",
    "amt","first","last","gender","street","city","state","zip",
    "lat","long","city_pop","job","dob","trans_num","unix_time",
    "merch_lat","merch_long","is_fraud","merch_zipcode"
]
SQL_KEYWORDS = {
    'select','from','where','group','by','order','having','limit','as',
    'and','or','not','in','is','null','on','join','inner','left','right',
    'outer','union','all','distinct','case','when','then','else','end',
    'sum','count','avg','min','max','coalesce','cast','between','like',
    'asc','desc','with','exists','any','top','offset','fetch','next',
    'rows','only','over','partition','int','double','text','bigint',
    'timestamp','date','boolean','true','false','transactions','max','min'
}
COLUMN_ALIASES = {
    "fraud_amount":"amt","amount":"amt","transaction_amount":"amt",
    "total_amount":"amt","card":"cc_num","card_num":"cc_num",
    "card_number":"cc_num","credit_card":"cc_num","fraud":"is_fraud",
    "is_fraudulent":"is_fraud","fraudulent":"is_fraud",
    "transaction_date":"trans_date_trans_time",
    "transaction_date_trans_time":"trans_date_trans_time",
    "date":"trans_date_trans_time","transaction_id":"trans_num",
    "zipcode":"merch_zipcode","zip_code":"zip",
    "latitude":"lat","longitude":"long","population":"city_pop",
}

def build_prompt(question):
    return (
        "### Instruction:\n"
        "You are an assistant that writes SQL queries for DuckDB.\n"
        "Only output a single SQL query, nothing else.\n\n"
        f"### Schema:\n{_SCHEMA_TEXT}\n\n"
        f"### Question:\n{question}\n\n"
        "### Response (SQL only):"
    )

def correct_column_names(sql):
    tokens = re.split(r'(\s+|,|\(|\)|;)', sql)
    result, skip_next = [], False
    for token in tokens:
        stripped = token.strip()
        if skip_next:
            result.append(token)
            if stripped: skip_next = False
            continue
        if stripped.upper() == 'AS':
            skip_next = True
            result.append(token)
            continue
        if (not stripped or stripped.lower() in SQL_KEYWORDS
                or re.match(r'^[\d\.]+$', stripped)
                or stripped.startswith("'") or stripped.startswith('"')):
            result.append(token)
            continue
        if re.match(r'^[a-zA-Z_][a-zA-Z0-9_]*$', stripped):
            lower = stripped.lower()
            if lower in COLUMN_ALIASES:
                token = token.replace(stripped, COLUMN_ALIASES[lower])
            elif stripped not in VALID_COLUMNS and lower not in SQL_KEYWORDS:
                close = get_close_matches(lower, VALID_COLUMNS, n=1, cutoff=0.6)
                if close:
                    token = token.replace(stripped, close[0])
        result.append(token)
    return ''.join(result)

def is_valid_sql(sql):
    try:
        test = f"SELECT * FROM ({sql.rstrip(';')}) __v__ LIMIT 0;"
        get_con().execute(test)
        return True
    except:
        return False

def extract_sql(raw):
    raw = re.sub(r'```(?:sql)?\s*', '', raw, flags=re.IGNORECASE).strip()
    lines = [l for l in raw.splitlines() if not l.strip().startswith("--")]
    raw = "\n".join(lines).strip()
    match = re.search(r'(SELECT\b[\s\S]*?;)', raw, re.IGNORECASE)
    if match: return match.group(1).strip()
    match2 = re.search(r'(SELECT\b[\s\S]*)', raw, re.IGNORECASE)
    if match2: return match2.group(1).strip().split("\n\n")[0].strip() + ";"
    return raw + ";"

def fix_sql(sql):
    sql = correct_column_names(sql)
    s = re.sub(r'\s+', ' ', sql).strip()
    if not s.upper().startswith("SELECT"): return sql
    where_match = re.search(r'\bWHERE\b(.*?)(?=\bGROUP\b|\bORDER\b|\bLIMIT\b|;|$)', s, re.IGNORECASE)
    if where_match:
        where_body = where_match.group(1).strip()
        conditions = re.split(r'\b(AND|OR)\b', where_body, flags=re.IGNORECASE)
        where_conds, having_conds = [], []
        for cond in conditions:
            if cond.upper().strip() in ('AND','OR'): continue
            if re.search(r'\b(SUM|COUNT|AVG|MIN|MAX)\s*\(', cond, re.IGNORECASE):
                having_conds.append(cond.strip())
            else:
                where_conds.append(cond.strip())
        new_where  = f"WHERE {' AND '.join(where_conds)} " if where_conds else ""
        new_having = f"HAVING {' AND '.join(having_conds)} " if having_conds else ""
        s = re.sub(r'\bWHERE\b.*?(?=\bGROUP\b|\bORDER\b|\bLIMIT\b|;|$)', new_where, s, flags=re.IGNORECASE)
        if new_having:
            s = re.sub(r'(?=\bORDER\s+BY\b|\bLIMIT\b|;|$)', new_having, s, count=1, flags=re.IGNORECASE)
    outer_select = re.match(r'^SELECT\s+(.*?)\s+FROM\s+\w+', s, re.IGNORECASE)
    has_agg      = bool(re.search(r'\b(SUM|COUNT|AVG|MIN|MAX)\s*\(', s, re.IGNORECASE))
    has_group_by = bool(re.search(r'\bGROUP\s+BY\b', s, re.IGNORECASE))
    if outer_select and has_agg:
        select_part = outer_select.group(1)
        bare_cols = []
        for col in select_part.split(','):
            col = col.strip()
            if re.match(r'(SUM|COUNT|AVG|MIN|MAX)\s*\(', col, re.IGNORECASE): continue
            col_name = re.split(r'\s+AS\s+', col, flags=re.IGNORECASE)[0].strip()
            if col_name and re.match(r'^[a-zA-Z_][a-zA-Z0-9_]*$', col_name):
                bare_cols.append(col_name)
        if bare_cols:
            if has_group_by:
                gb_match = re.search(r'\bGROUP\s+BY\s+(.*?)(?=\bORDER\b|\bHAVING\b|\bLIMIT\b|;|$)', s, re.IGNORECASE)
                if gb_match:
                    existing = [g.strip() for g in gb_match.group(1).split(',')]
                    missing  = [c for c in bare_cols if c.lower() not in [e.lower() for e in existing]]
                    if missing:
                        s = re.sub(r'\bGROUP\s+BY\s+.*?(?=\bORDER\b|\bHAVING\b|\bLIMIT\b|;|$)',
                                   f'GROUP BY {", ".join(existing + missing)} ', s, flags=re.IGNORECASE)
            else:
                s = re.sub(r'(?=\bORDER\s+BY\b|\bLIMIT\b|;|$)',
                           f' GROUP BY {", ".join(bare_cols)} ', s, count=1, flags=re.IGNORECASE)
    return s.strip().rstrip(';') + ';'

def generate_sql(question, max_new_tokens=256, max_retries=3):
    prompt = build_prompt(question)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    input_len = inputs["input_ids"].shape[1]
    for attempt in range(max_retries):
        temp = 0.1 + (attempt * 0.2)
        with torch.no_grad():
            out = model.generate(
                **inputs, max_new_tokens=max_new_tokens, min_new_tokens=10,
                do_sample=True, temperature=temp, top_p=0.95,
                repetition_penalty=1.1, pad_token_id=tokenizer.eos_token_id)
        raw = tokenizer.decode(out[0][input_len:], skip_special_tokens=True).strip()
        sql = fix_sql(extract_sql(raw))
        if is_valid_sql(sql):
            return sql
    return sql

def choose_axes(df, question):
    if df.empty: return None, None, None
    cols = list(df.columns)
    num  = [c for c in cols if is_numeric_dtype(df[c])]
    dts  = [c for c in cols if is_datetime64_any_dtype(df[c])]
    cat  = [c for c in cols if df[c].dtype == "object"]
    q    = question.lower()
    tq   = any(k in q for k in ["over time","by day","by month","trend","daily","monthly"])
    if tq and not dts:
        for c in cols:
            if "date" in c.lower() or "time" in c.lower():
                try: df[c] = pd.to_datetime(df[c]); dts.append(c)
                except: pass
    if tq and dts and num: return dts[0], next((c for c in num if "id" not in c.lower()), num[0]), "line"
    if cat and num:        return cat[0], next((c for c in num if c != cat[0]), num[0]), "bar"
    if len(num) >= 2:      return num[0], num[1], "scatter"
    return None, None, None

def create_plot(df, question):
    if df is None or df.empty: return None
    x, y, kind = choose_axes(df, question)
    if not all([x, y, kind]): return None
    dp = df[[x, y]].dropna()
    if dp.empty: return None
    if len(dp) > 500: dp = dp.sample(500, random_state=42)
    fig, ax = plt.subplots()
    if kind == "line":
        dp = dp.sort_values(x); ax.plot(dp[x], dp[y])
    elif kind == "bar":
        if dp[x].nunique() > 20:
            dp = dp.groupby(x, as_index=False)[y].sum().sort_values(y, ascending=False).head(20)
        ax.bar(dp[x], dp[y]); ax.set_xticklabels(dp[x], rotation=45, ha="right")
    elif kind == "scatter":
        ax.scatter(dp[x], dp[y])
    ax.set_xlabel(x); ax.set_ylabel(y); ax.set_title(question)
    fig.tight_layout()
    return fig

def query_agent_with_viz(question):
    if not question.strip(): return "", pd.DataFrame(), None, "Please enter a question."
    try:
        sql = generate_sql(question)
    except Exception as e:
        return "", pd.DataFrame(), None, f"SQL generation error: {e}"
    if not sql.strip().upper().startswith("SELECT"):
        return sql, pd.DataFrame(), None, f"Generated SQL did not start with SELECT. Got: {sql[:120]!r}"
    try:
        df_result = get_con().execute(sql).fetchdf()
    except Exception as e:
        return sql, pd.DataFrame(), None, f"SQL execution error: {e}"
    try:
        fig = create_plot(df_result, question)
    except Exception as e:
        return sql, df_result, None, f"Query ran; visualization failed: {e}"
    return sql, df_result, fig, ""

def on_submit(question):
    sql, df_result, fig, error_msg = query_agent_with_viz(question)
    if error_msg:
        return sql, pd.DataFrame({"Error": [error_msg]}), None, f"❌ {error_msg}"
    return sql, df_result, fig, "✅ Success"

try: demo.close()
except: pass

with gr.Blocks(title="LYNX AI — Credit Card Analytics") as demo:
    gr.Markdown("# 💳 LYNX AI — Fraud Detection & Transaction Analytics")
    gr.Markdown("Ask questions in plain English. SQL is generated automatically, results shown in a table, charts drawn when relevant.")
    question_box = gr.Textbox(label="Your question", placeholder="e.g., Which merchant has the most fraudulent transactions?", lines=2)
    run_btn   = gr.Button("Run query")
    sql_box   = gr.Textbox(label="Generated SQL", interactive=False)
    result_df = gr.Dataframe(label="Query result", interactive=False)
    plot_out  = gr.Plot(label="Visualization")
    error_box = gr.Markdown()
    run_btn.click(fn=on_submit, inputs=[question_box], outputs=[sql_box, result_df, plot_out, error_box])
    gr.Examples(
        examples=[
            "Which merchant has the most fraudulent transactions?",
            "Show the top 10 merchants by total fraud amount.",
            "Plot the fraud rate by transaction category.",
            "How many fraudulent transactions are there per state?",
            "What is the average transaction amount by category?",
        ],
        inputs=[question_box], outputs=[sql_box, result_df, plot_out, error_box],
        fn=on_submit, cache_examples=False,
    )

demo.launch(debug=True)


Loading NL->SQL model from ./nl2sql-cc-finetuned...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

✅ NL->SQL model ready
✅ DuckDB connected
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://f5d281bd9e513493b2.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


/tmp/ipython-input-19509/1622507287.py:264: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.bar(dp[x], dp[y]); ax.set_xticklabels(dp[x], rotation=45, ha="right")
